# Part 2.3 (b) — Stability: does the same input give the same answer?

NVIDIA was run three times on 2026-10-10 with identical arguments (`scripts/run_pipeline.py NVDA ... --tag stab1/2/3`),
minutes apart. Compared: the page-1 rating and target, the valuation engine's target, the top-3 risks, and the grounding
of every section (Part 2.3 scorer, plus the LLM-free status-error counter).

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "audit" else Path.cwd()
sys.path.insert(0, str(ROOT / "audit"))
import grounding as g
from openai import OpenAI
from modules.common_utils import load_config, get_api_key

pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 90)
RUNS = {f"stab{i}": g.OUT / "NVDA" / f"stab{i}" for i in (1, 2, 3)}
client = OpenAI(api_key=get_api_key(load_config(str(g.SRC.parent / "config" / "config.ini")),
                                    section="API_KEYS", key="openai_api_key"), max_retries=8)
summary = pd.read_csv(ROOT / "notes" / "run_summary.csv").set_index("run")
summary.loc[[f"NVDA/{r}" for r in RUNS],
            ["price", "page1_rating", "page1_target", "fmp_grade", "engine_target", "engine_up_down"]]

,price,page1_rating,page1_target,fmp_grade,engine_target,engine_up_down
run,,,,,,
NVDA/stab1,$229.28,Buy,$339.17,Overweight,$82.24,-64.1%
NVDA/stab2,$229.28,Buy,$339.17,Overweight,$82.24,-64.1%
NVDA/stab3,$229.28,Buy,$339.17,Overweight,$82.24,-64.1%


Every page-1 number is identical across runs because none of them comes from the LLM: price, consensus target and
broker grade are fetched from FMP, the rating is a fixed rule on target vs. price, and the engine is a closed-form
function of the forecast table (Part 2.2). They are stable by construction, so a stable headline says nothing about
whether the text underneath agrees with it.

In [2]:
def top_risks(text: str, n: int = 5) -> list[str]:
    heads = re.findall(r"^\s*(?:[-*\u2022]|\d+\.)\s*\*\*([^*]+?)\*\*", text, re.M)
    return [h.strip(" :") for h in heads][:n]


CATEGORIES = {  # keyword -> risk category, so differently worded headings can be compared
    "valuation": "valuation", "growth": "growth sustainability", "sustainab": "growth sustainability",
    "margin": "margins / profitability", "profitab": "margins / profitability",
    "compet": "competition", "peer": "competition", "execution": "execution / operations",
    "operational": "execution / operations", "concentration": "concentration / cyclicality",
    "cyclical": "concentration / cyclicality", "demand": "concentration / cyclicality",
}


def category(heading: str) -> str:
    h = heading.lower()
    return next((c for k, c in CATEGORIES.items() if k in h), "other: " + heading)


risks = {r: top_risks((d / "analysis" / "risks.txt").read_text(encoding="utf-8")) for r, d in RUNS.items()}
pd.DataFrame({r: [f"{h}  [{category(h)}]" for h in hs] for r, hs in risks.items()},
             index=[f"risk {i}" for i in range(1, 6)])

,stab1,stab2,stab3
risk 1,Valuation Risk [valuation],Valuation Risk [valuation],Valuation Risk [valuation]
risk 2,Sustainability of High Growth [growth sustainability],Sustainability of Revenue Growth [growth sustainability],Revenue Growth Sustainability [growth sustainability]
risk 3,Margin Compression Risk [margins / profitability],Peer Competition and Market Share Risk [competition],Competitive Pressure [competition]
risk 4,Peer Competitive Threats [competition],Margin Compression Possibility [margins / profitability],Profitability Volatility [margins / profitability]
risk 5,Execution and Concentration Risk [execution / operations],Execution and Operational Risks [execution / operations],Execution and Operational Risk [execution / operations]


In [3]:
def jaccard(a, b):
    return len(a & b) / len(a | b)


cats = {r: [category(h) for h in hs] for r, hs in risks.items()}
pairs = [("stab1", "stab2"), ("stab1", "stab3"), ("stab2", "stab3")]
print("top-3 identical as ordered lists:", len({tuple(c[:3]) for c in cats.values()}) == 1)
print("top-3 category overlap (Jaccard):", {f"{a}-{b}": round(jaccard(set(cats[a][:3]), set(cats[b][:3])), 2) for a, b in pairs})
print("all-5 category overlap (Jaccard):", {f"{a}-{b}": round(jaccard(set(cats[a]), set(cats[b])), 2) for a, b in pairs})

top-3 identical as ordered lists: False
top-3 category overlap (Jaccard): {'stab1-stab2': 0.5, 'stab1-stab3': 0.5, 'stab2-stab3': 1.0}
all-5 category overlap (Jaccard): {'stab1-stab2': 1.0, 'stab1-stab3': 1.0, 'stab2-stab3': 1.0}


## Grounding across runs

Same scorer as Part 2.3, applied to each run's seven Step 1 sections and the two sections the report regenerates.

In [4]:
rows = []
for r, d in RUNS.items():
    claims, sections = g.score_run(client, "gpt-4.1", "NVDA", "NVIDIA Corporation", run_dir=d, skip=("news_summary",))
    tables = g.load_tables(d)
    for (s, v), grp in claims.groupby(["section", "variant"]):
        text = sections.query("section == @s and variant == @v")["text"].iloc[0]
        vc = grp["verdict"].value_counts()
        rows.append({"run": r, "section": s, "variant": v, "claims": len(grp),
                     "supported": vc.get("Supported", 0), "contradicted": vc.get("Contradicted", 0),
                     "status errors (counter)": g.status_errors(text, tables)["total"],
                     "words": len(text.split())})
stab = pd.DataFrame(rows)
stab["grounding rate"] = stab["supported"] / stab["claims"]
stab.to_csv(ROOT / "audit" / "stability_grounding.csv", index=False)
wide = stab.pivot_table(index=["section", "variant"], columns="run", values="grounding rate")
wide["range (max-min)"] = wide.max(axis=1) - wide.min(axis=1)
wide.round(2)

run                          stab1  stab2  stab3  range (max-min)
section             variant                                      
company_overview    step1     0.21   0.29   0.33             0.12
competitor_analysis report    0.55   0.20   0.30             0.35
                    step1     0.60   0.55   0.88             0.33
investment_overview step1     0.58   0.39   0.55             0.19
major_takeaways     report    1.00   0.67   0.00             1.00
                    step1     0.50   0.50   0.50             0.00
risks               step1     0.40   0.57   0.44             0.17
tagline             step1     1.00   0.75   0.50             0.50
valuation_overview  step1     0.27   0.44   0.25             0.19

In [5]:
s1 = stab[stab.variant == "step1"].groupby("run")[["claims", "supported", "contradicted", "status errors (counter)"]].sum()
s1["grounding rate"] = s1["supported"] / s1["claims"]
s1["contradicted rate"] = s1["contradicted"] / s1["claims"]
s1.round(3)

,claims,supported,contradicted,status errors (counter),grounding rate,contradicted rate
run,,,,,,
stab1,63,27,16,14,0.429,0.254
stab2,70,31,16,13,0.443,0.229
stab3,67,31,14,8,0.463,0.209


## Findings

1. **The headline is perfectly stable, and that is uninformative.** Rating (Buy), target ($339.17), broker grade and
   the engine's $82.24 are identical in all three runs, because none of them passes through the LLM. They come from FMP
   and from fixed formulas.
2. **The risk list is stable as a set, not as a ranking.** All three runs name the same five risk categories (Jaccard
   1.0). Valuation and growth sustainability are always #1 and #2, but #3 alternates between margins and competition,
   so the top-3 overlap is 0.5 / 0.5 / 1.0. A "top-3 risks" output is not reproducible.
3. **Error rates are stable; individual errors are not.** Over the seven Step 1 sections the grounding rate is
   43% / 44% / 46% and the contradicted rate 25% / 23% / 21%; the LLM-free counter finds 14 / 13 / 8 status errors.
   Section by section the same input gives very different results: the tagline is 100% / 75% / 50% grounded, the
   Step 1 competitor analysis 60% / 55% / 88%, and the report's gpt-4o-mini takeaways 100% / 67% / 0%.
4. **Implication.** At temperature 0.7 one run is one sample. Any section-level claim about quality needs several
   samples, which is why Part 3 is evaluated on 70 company-sections per condition. Because the error *rate* is a
   property of the prompt and the evidence, the fix belongs there, not in patching individual outputs.
